In [1]:
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import trustworthiness
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import math
import json
import os


In [2]:
os.environ["LOKY_MAX_CPU_COUNT"] = '14'
os.environ["OMP_NUM_THREADS"] = "1"

In [3]:
bronze = pd.read_csv("processed_csvs/bronze.csv")
champion = pd.read_csv("processed_csvs/champion.csv")
diamond = pd.read_csv("processed_csvs/diamond.csv")
gold = pd.read_csv("processed_csvs/gold.csv")
grand = pd.read_csv("processed_csvs/grand.csv")
platinum = pd.read_csv("processed_csvs/platinum.csv")
silver = pd.read_csv("processed_csvs/silver.csv")
supersonic = pd.read_csv("processed_csvs/supersonic.csv")

ranks = {"Bronze": bronze, "Champion": champion, "Diamond": diamond, "Gold": gold, "Grandchampion": grand, 
         "Platinum": platinum, "Silver": silver, "Supersonic": supersonic}

low = pd.concat([bronze, silver, gold], ignore_index=True)
intermediate = pd.concat([platinum, diamond, champion], ignore_index=True)
high = pd.concat([grand, supersonic], ignore_index=True)

rank_levels = {'Low': low, 'Intermediate': intermediate, 'High': high}

In [4]:
for rank, df in rank_levels.items():
    rank_levels[rank] = df.dropna()

rank_sne = {}

for rank, df in rank_levels.items():
    temp = df.drop(columns=[
    'color','player name','platform','player id','car id','car name','rank'])
    rank_sne[rank] = temp

for rank, df in rank_sne.items():
    check = df.var().sort_values()
    for j in check.index:
        if check[j] == 0:
            print(j)

for rank, df in rank_sne.items():
    print(df.isna().sum().sort_values())

game duration                 0
score                         0
goals                         0
assists                       0
saves                         0
                             ..
percentage neutral third      0
time offensive third          0
percentage offensive third    0
demos inflicted               0
demos taken                   0
Length: 66, dtype: int64
game duration                 0
score                         0
goals                         0
assists                       0
saves                         0
                             ..
percentage neutral third      0
time offensive third          0
percentage offensive third    0
demos inflicted               0
demos taken                   0
Length: 66, dtype: int64
game duration                 0
score                         0
goals                         0
assists                       0
saves                         0
                             ..
percentage neutral third      0
time offensive third  

In [5]:
tsne_data_raw = {}

for rank, df in rank_sne.items():
    scaler = StandardScaler()
    rank_scaled = scaler.fit_transform(df)

    rank_tsne = TSNE(n_components=2, random_state=42, perplexity=10).fit_transform(rank_scaled)

    tsne_data_raw[rank] = rank_tsne

In [ ]:
#exports tsne_data_raw to a flat list of points with rank labels for website visuals 

points = []

for rank, coords in tsne_data_raw.items():
    for x, y in coords:
        points.append({
            "rank": rank,
            "x": float(x),
            "y": float(y)
        })

with open("tsne_points.json", "w") as f:
    json.dump(points, f)